In [ ]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import random
from Functions.Utils import *
from Functions.Graphs import *
random.seed(101)
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

path = r'Datasets\DEVRT\NISSAN_LEAF_TRAIN'
samples = os.listdir(path)

for i,sample in enumerate(samples):
    df = pd.read_csv(os.path.join(path,sample))
    pwr = (df['Motor Pwr(w)'].values)
    spd = (df['speed'].values)
    rpm = (df['rpm'].values)
    trq = (df['Torque Nm'].values)
    elv = (df['elv_spy'].values)
    lat = (df['latitude'].values)
    lon = (df['longitude'].values)
    alt = (df['altitude'].values)
    df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv,'lat': lat, 'lon': lon, 'alt': alt, 'power': pwr,})
    #df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'power': pwr,})
    if i == 0:
        df_train = df
    else:
        df_train = pd.concat([df_train,df])

path = r'Datasets\DEVRT\NISSAN_LEAF_TEST'
samples = os.listdir(path)
samples_test = random.sample(samples, int(len(samples)*0.5))
samples_val = list(set(samples) - set(samples_test))

        

In [30]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# --- 1. SETUP HYPERPARAMETERS ---
N_input = 50     # Widened: 50 steps * 7s = ~5.8 minutes of historical context
N_p = 5          # 5 steps ahead * 7s = 35 seconds prediction horizon
batch_size = 256  
epochs = 60      
hidden_n = 128
num_layers = 3
learning_rate = 0.0005 # Lowered to smooth out convergence and prevent early flatlining

# --- 2. LOAD SEPARATE TRIP LISTS ---
path_train = r'Datasets\DEVRT\NISSAN_LEAF_TRAIN'
samples_train = [os.path.join(path_train, s) for s in os.listdir(path_train)]

path_test_dir = r'Datasets\DEVRT\NISSAN_LEAF_TEST'
all_test_samples = os.listdir(path_test_dir)
samples_test_names = random.sample(all_test_samples, int(len(all_test_samples) * 0.5))
samples_val_names = list(set(all_test_samples) - set(samples_test_names))

samples_test = [os.path.join(path_test_dir, s) for s in samples_test_names]
samples_val = [os.path.join(path_test_dir, s) for s in samples_val_names]

def load_trip_data(file_paths):
    trips = []
    for path in file_paths:
        df = pd.read_csv(path)
        df = df.ffill().bfill()
        
        speed_raw = df['speed'].values
        speed_delta = np.diff(speed_raw, prepend=speed_raw[0])
        accel_trend = pd.Series(speed_delta).rolling(window=3, min_periods=1).mean().values
        
        alt_raw = df['altitude'].values
        alt_delta = np.diff(alt_raw, prepend=alt_raw[0])
        
        trip_df = pd.DataFrame({
            'speed': speed_raw, 
            'speed_delta': speed_delta,   
            'accel_trend': accel_trend,   
            'rpm': df['rpm'].values, 
            'torque': df['Torque Nm'].values, 
            'elv': df['elv_spy'].values,
            'alt_delta': alt_delta,       
            #'max_speed': df['max_speed'].values,       
            #'soc': df['soc'].values,                   
            #'frontal_wind': df['Frontal_Wind'].values, 
            #'aux_pwr': df['Aux Pwr(100w)'].values,     
            'power': df['Motor Pwr(w)'].values,
            #'power': exponential_moving_average(df['Motor Pwr(w)'],alpha=0.5).values
        })
        #trip_df = moving_average_df(trip_df,3)
        trip_df = exponential_moving_average_df(trip_df,0.3)
        trips.append(trip_df)
    return trips

trips_train = load_trip_data(samples_train)
trips_val = load_trip_data(samples_val)
trips_test = load_trip_data(samples_test)

# --- 3. FIT SCALERS ON TRAINING DATA ONLY ---
df_all_train = pd.concat(trips_train, axis=0)
min_power = df_all_train['power'].min()
power_shift = abs(min_power) if min_power < 0 else 0

for t_list in [trips_train, trips_val, trips_test]:
    for trip in t_list:
        trip['power'] = np.log1p(trip['power'] + power_shift)

df_all_train_transformed = pd.concat(trips_train, axis=0)
scaler_X = StandardScaler().fit(df_all_train_transformed.values[:, :-1])
num_features = df_all_train_transformed.shape[1] - 1

# --- 4. WINDOW SLIDING FUNCTION ---
def create_dataset_sequences(trips_list, scaler_x, n_in, n_out):
    X, Y = [], []
    for trip in trips_list:
        data = trip.values
        if len(data) < (n_in + n_out):
            continue
        features_scaled = scaler_x.transform(data[:, :-1])
        power_log = data[:, -1]
        full_features = np.hstack((features_scaled, power_log.reshape(-1, 1)))
        
        for k in range(len(data) - n_in - n_out + 1):
            X.append(full_features[k : k + n_in])
            Y.append(power_log[k + n_in : k + n_in + n_out])
            
    return np.array(X), np.array(Y)

X_train, Y_train = create_dataset_sequences(trips_train, scaler_X, N_input, N_p)
X_val, Y_val = create_dataset_sequences(trips_val, scaler_X, N_input, N_p)
X_test, Y_test = create_dataset_sequences(trips_test, scaler_X, N_input, N_p)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(Y_test, dtype=torch.float32)), batch_size=batch_size, shuffle=False)

# --- 5. DEFINE MODEL ---
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super(LSTMForecaster, self).__init__()
        # Increased dropout to 0.4 to combat early track memorization
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.4)
        self.ln = nn.LayerNorm(hidden_dim)
        # Added a secondary dropout barrier before the prediction head
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_time_step = lstm_out[:, -1, :] 
        normalized_features = self.ln(last_time_step)
        return self.fc(self.drop(normalized_features))

model = LSTMForecaster(input_dim=num_features + 1, hidden_dim=hidden_n, output_dim=N_p, num_layers=num_layers)
criterion = nn.HuberLoss(delta=2.0)
#criterion = nn.MSELoss()
# Reintroduced subtle weight decay to keep weights bounded
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5) 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- 6. TRAINING & VALIDATION LOOP WITH CHECKPOINTING ---
print("Starting training...")
best_val_loss = float('inf')

for epoch in range(epochs):
    # Training Phase
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_x.size(0)
    avg_train_loss = train_loss / len(X_train)
    
    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = criterion(model(batch_x), batch_y)
            val_loss += loss.item() * batch_x.size(0)
    avg_val_loss = val_loss / len(X_val)
    
    scheduler.step(avg_val_loss)
    
    # Save a model checkpoint only if the validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_lstm_model.pth')
        checkpoint_msg = "--> Best Model Saved!"
    else:
        checkpoint_msg = ""
        
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} {checkpoint_msg}")

# --- 7. RELOAD OPTIMAL WEIGHTS & EVALUATE ON TEST SET ---
print("\nReloading best historical model configurations for evaluation...")
model.load_state_dict(torch.load('best_lstm_model.pth'))

model.eval()
test_loss = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        loss = criterion(model(batch_x), batch_y)
        test_loss += loss.item() * batch_x.size(0)
avg_test_loss = test_loss / len(X_test)

print("\n==============================")
print(f"Final Out-of-Sample Test Huber Loss: {avg_test_loss:.4f}")
print("==============================\n")

# --- 8. INFERENCE VISUALIZATION (From Test Set) ---
with torch.no_grad():
    test_sample_in = torch.tensor(X_test[100:101], dtype=torch.float32)
    pred_log = model(test_sample_in).numpy().flatten()
    actual_log = Y_test[100:101].flatten()
    
    predicted_power = np.expm1(pred_log) - power_shift
    actual_power = np.expm1(actual_log) - power_shift
    predicted_power = np.clip(predicted_power, 0, None)
    
    print("--- Real Test Set Example Prediction ---")
    print(f"Predicted Motor Power: {np.round(predicted_power, 1)}")
    print(f"Actual Motor Power:    {np.round(actual_power, 1)}")

Starting training...
Epoch 01/60 | Train Loss: 11.0834 | Val Loss: 8.4804 --> Best Model Saved!
Epoch 02/60 | Train Loss: 7.5605 | Val Loss: 6.7250 --> Best Model Saved!
Epoch 03/60 | Train Loss: 5.8873 | Val Loss: 5.0542 --> Best Model Saved!
Epoch 04/60 | Train Loss: 4.2130 | Val Loss: 3.4068 --> Best Model Saved!
Epoch 05/60 | Train Loss: 2.7244 | Val Loss: 1.9764 --> Best Model Saved!
Epoch 06/60 | Train Loss: 1.6306 | Val Loss: 1.0316 --> Best Model Saved!
Epoch 07/60 | Train Loss: 1.0202 | Val Loss: 0.6513 --> Best Model Saved!
Epoch 08/60 | Train Loss: 0.8856 | Val Loss: 0.5821 --> Best Model Saved!
Epoch 09/60 | Train Loss: 0.8399 | Val Loss: 0.5608 --> Best Model Saved!
Epoch 10/60 | Train Loss: 0.7736 | Val Loss: 0.4430 --> Best Model Saved!
Epoch 11/60 | Train Loss: 0.6935 | Val Loss: 0.3842 --> Best Model Saved!
Epoch 12/60 | Train Loss: 0.6569 | Val Loss: 0.3256 --> Best Model Saved!
Epoch 13/60 | Train Loss: 0.6117 | Val Loss: 0.2979 --> Best Model Saved!
Epoch 14/60 | Tr

C:\Users\Claudio\AppData\Local\Temp\ipykernel_24340\1521212224.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_lstm_model.pth'))

In [31]:
# --- 8. ROW-BY-ROW ITERATIVE STREAMING FUNCTION DEFINITION ---
# Initialize persistent variables for streaming tracking
h_step = torch.zeros(num_layers, 1, hidden_n, device=device)
c_step = torch.zeros(num_layers, 1, hidden_n, device=device)
last_predicted_log_power = 0.0

def pred_single_row(features_raw):
    """
    Accepts a single row of features at a single timestamp, scales it, 
    injects the last known power feedback, and updates the global LSTM states.
    """
    global h_step, c_step, last_predicted_log_power
    
    # 1. Shape to 2D array for scaler compatibility -> [1, num_features]
    x_arr = np.array(features_raw, dtype=np.float32).reshape(1, -1)
    x_scaled = scaler_X.transform(x_arr)
    
    # 2. Append the target layer prediction from the previous step
    full_x_row = np.hstack((x_scaled, np.array([[last_predicted_log_power]], dtype=np.float32)))
    
    # 3. Shape into standard PyTorch Recurrent structure -> [Batch=1, Time_Step=1, Features]
    x_tensor = torch.tensor(full_x_row, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # 4. Process step and capture updated context tuple
        pred_scaled, (h_step, c_step) = model(x_tensor, (h_step, c_step))
        pred_log = pred_scaled.cpu().numpy().flatten()
        
    # 5. Buffer the current output for the next row's past feature assignment
    last_predicted_log_power = pred_log[0]
    
    # 6. Revert Log Transform and Shift back into physical Watts
    predicted_power = np.expm1(pred_log) - power_shift
    return np.clip(predicted_power, 0, None)

# --- 9. STREAMING SIMULATION PIPELINE RUN ---
# Select the first raw test file path to simulate real-time processing
stream_file_path = samples_val[1]
df_stream = pd.read_csv(stream_file_path).ffill().bfill()

# Reconstruct features identically to the loading loop parameters
speed_raw = df_stream['speed'].values
speed_delta = np.diff(speed_raw, prepend=speed_raw[0])
accel_trend = pd.Series(speed_delta).rolling(window=3, min_periods=1).mean().values
alt_raw = df_stream['altitude'].values
alt_delta = np.diff(alt_raw, prepend=alt_raw[0])

trip_df_stream = pd.DataFrame({
    'speed': speed_raw, 
    #'speed_delta': speed_delta,   
    #'accel_trend': accel_trend,   
    'rpm': df_stream['rpm'].values, 
    'torque': df_stream['Torque Nm'].values, 
    #'elv': df_stream['elv_spy'].values,
    #'alt_delta': alt_delta,       
    'power': df_stream['Motor Pwr(w)'].values
})
trip_df_stream = exponential_moving_average_df(trip_df_stream, 0.3)

# Extract inputs and real targets
stream_features = trip_df_stream.drop(columns=['power']).values
#actual_power_profile = df_stream['Motor Pwr(w)'].values
actual_power_profile = trip_df_stream['power'].values

print(f"\n==============================")
print(f"Simulating Sequential Processing Over {len(stream_features)} Rows...")
print(f"==============================\n")

# Reset memory vectors before beginning the trip
h_step = torch.zeros(num_layers, 1, hidden_n, device=device)
c_step = torch.zeros(num_layers, 1, hidden_n, device=device)
last_predicted_log_power = 0.0

yR, yP = [], []

for idx in range(len(stream_features)-N_p):
    current_timestamp_features = stream_features[idx]
    
    # Dynamic Step Call
    horizon_forecast = pred_single_row(current_timestamp_features)
    horizon_real = actual_power_profile[idx : idx + N_p]
    if idx == 0:
        yR = horizon_real.reshape(-1,N_p)
        yP = horizon_forecast.reshape(-1,N_p)
    else:
        yR = np.vstack((yR,horizon_real))
        yP = np.vstack((yP,horizon_forecast))

yR = yR.T
yP = yP.T

i=2
PlotSeriesPLY(ySeries=[yR[i],yP[i]])

NameError: name 'device' is not defined